# Rigorous A/B Test Analysis: Beyond Statistical Significance
### A Product Experiment Case Study | Shlok Sheth

---

## Business Problem

A product team built a new engagement feature and wants to know: **should we launch it?**

A naive analyst checks p < 0.05 and calls it done. This notebook shows what rigorous experiment analysis actually looks like:

| Step | What we check |
|------|---------------|
| Pre-experiment | Sample ratio mismatch, covariate balance, power analysis |
| Primary metric | Sessions per user, with CUPED variance reduction |
| Guardrail metrics | Conversion rate, support tickets, unsubscribe rate |
| Segmentation | Heterogeneous treatment effects by age and platform |
| Decision | Launch recommendation with explicit tradeoff reasoning |

---

**Dataset:** 50,000 users (25,000 per group), synthetic but realistic  
**Author:** Shlok Sheth | sheth53@purdue.edu | https://shethshlok.netlify.app

---
## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0D1117', 'axes.facecolor': '#0D1117',
    'axes.edgecolor': '#30363D', 'axes.labelcolor': '#C9D1D9',
    'xtick.color': '#8B949E', 'ytick.color': '#8B949E',
    'text.color': '#C9D1D9', 'grid.color': '#21262D', 'grid.alpha': .6,
    'font.family': 'monospace', 'axes.spines.top': False, 'axes.spines.right': False,
})
BLUE='#58A6FF'; ORANGE='#F78166'; GREEN='#3FB950'
GOLD='#E3B341'; PURPLE='#BC8CFF'; BG='#0D1117'; CARD='#161B22'

print('Setup complete.')

---
## 1. Pre-Experiment: Power Analysis

Before running any experiment, we answer: **how many users do we need to detect a meaningful effect?**

Key decisions made upfront:
- **Alpha (Type I error):** 0.05 — 5% false positive rate
- **Power (1 - Type II error):** 80% — 80% chance of detecting a real effect
- **MDE (Minimum Detectable Effect):** 2.5pp — the smallest lift that would justify launch costs
- **Baseline conversion rate:** measured from historical data

In [ ]:
# Load data
df   = pd.read_csv('../data/ab_test_data.csv')
ctrl = df[df['variant']=='control']
trt  = df[df['variant']=='treatment']

print(f'Total users:      {len(df):,}')
print(f'Control:          {len(ctrl):,}')
print(f'Treatment:        {len(trt):,}')
print(f'Baseline conv:    {ctrl["converted"].mean():.4f}')
print(f'Baseline sessions:{ctrl["sessions"].mean():.3f}')

In [ ]:
def required_sample_size(baseline, mde, alpha=0.05, power=0.80):
    """Two-proportion z-test sample size formula."""
    p1, p2   = baseline, baseline + mde
    z_alpha  = stats.norm.ppf(1 - alpha/2)
    z_beta   = stats.norm.ppf(power)
    pooled   = (p1 + p2) / 2
    n = (z_alpha * np.sqrt(2*pooled*(1-pooled)) +
         z_beta  * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2 / mde**2
    return int(np.ceil(n))

baseline_cr = ctrl['converted'].mean()
mde         = 0.025   # 2.5 percentage points
n_required  = required_sample_size(baseline_cr, mde)

print('=' * 50)
print(f'  Baseline conversion rate: {baseline_cr:.4f}')
print(f'  MDE:                      {mde:.3f} ({mde*100:.1f}pp)')
print(f'  Required per group:       {n_required:,}')
print(f'  Our sample per group:     {len(ctrl):,}')
print(f'  Status:                   {"Adequately powered" if len(ctrl) >= n_required else "UNDERPOWERED"}')
print('=' * 50)

---
## 2. Sample Ratio Mismatch (SRM) Check

An SRM occurs when the actual traffic split differs significantly from the intended split. It is one of the most common and dangerous bugs in experiment infrastructure, it invalidates all downstream results.

**Test:** Chi-square goodness of fit against the expected 50/50 split.

In [ ]:
observed   = [len(ctrl), len(trt)]
expected   = [len(df)/2, len(df)/2]
chi2, p_srm = stats.chisquare(observed, expected)

print('Sample Ratio Mismatch Test')
print('=' * 40)
print(f'  Control:    {len(ctrl):,} ({len(ctrl)/len(df):.3%})')
print(f'  Treatment:  {len(trt):,} ({len(trt)/len(df):.3%})')
print(f'  Chi2:       {chi2:.4f}')
print(f'  p-value:    {p_srm:.4f}')
print(f'  Result:     {"NO SRM detected" if p_srm > 0.01 else "WARNING: SRM detected: stop the experiment"}')
print('=' * 40)
print('\nIf SRM is detected, do NOT analyze the experiment.')
print('Investigate the assignment mechanism before proceeding.')

---
## 3. Covariate Balance Check

Even with proper randomization, we verify that pre-experiment covariates are balanced between groups. Imbalance indicates a problem with the randomization mechanism.

In [ ]:
# Continuous covariates: KS test
for col in ['days_active', 'prior_sessions']:
    ks_stat, ks_p = stats.ks_2samp(ctrl[col], trt[col])
    print(f'{col:<20} KS stat: {ks_stat:.4f}   p-value: {ks_p:.4f}   {"OK" if ks_p > 0.05 else "IMBALANCED"}')

# Categorical covariates: chi-square
for col in ['user_age', 'platform']:
    counts_c = ctrl[col].value_counts().sort_index()
    counts_t = trt[col].value_counts().sort_index()
    chi2, p = stats.chisquare(counts_t, counts_c * len(trt)/len(ctrl))
    print(f'{col:<20} Chi2:    {chi2:.4f}   p-value: {p:.4f}   {"OK" if p > 0.05 else "IMBALANCED"}')

---
## 4. Primary Metric Analysis with CUPED

**Primary metric:** Sessions per user

**CUPED (Controlled-experiment Using Pre-Experiment Data)** reduces variance by removing variation explained by a pre-experiment covariate (prior sessions). This increases statistical power without increasing sample size.

**Formula:** `Y_cuped = Y - theta * (X - E[X])`

where `theta = Cov(Y, X) / Var(X)` and `X` is the pre-experiment covariate.

In [ ]:
# Naive estimate
naive_lift    = trt['sessions'].mean() - ctrl['sessions'].mean()
t_naive, p_naive = stats.ttest_ind(trt['sessions'], ctrl['sessions'])
se_naive      = np.sqrt(ctrl['sessions'].var()/len(ctrl) + trt['sessions'].var()/len(trt))
ci_naive      = (naive_lift - 1.96*se_naive, naive_lift + 1.96*se_naive)

# CUPED estimate
grand_mean_prior = df['prior_sessions'].mean()
theta = (np.cov(ctrl['sessions'], ctrl['prior_sessions'])[0,1] / np.var(ctrl['prior_sessions']) +
         np.cov(trt['sessions'],  trt['prior_sessions'])[0,1]  / np.var(trt['prior_sessions'])) / 2

ctrl_cuped = ctrl['sessions'] - theta * (ctrl['prior_sessions'] - grand_mean_prior)
trt_cuped  = trt['sessions']  - theta * (trt['prior_sessions']  - grand_mean_prior)

cuped_lift    = trt_cuped.mean() - ctrl_cuped.mean()
t_cuped, p_cuped = stats.ttest_ind(trt_cuped, ctrl_cuped)
se_cuped      = np.sqrt(ctrl_cuped.var()/len(ctrl_cuped) + trt_cuped.var()/len(trt_cuped))
ci_cuped      = (cuped_lift - 1.96*se_cuped, cuped_lift + 1.96*se_cuped)
var_reduction = (1 - ctrl_cuped.var()/ctrl['sessions'].var()) * 100

print('Primary Metric: Sessions per User')
print('=' * 58)
print(f'  {"Method":<20} {"Lift":>8}  {"p-value":>10}  {"95% CI"}')
print('-' * 58)
print(f'  {"Naive t-test":<20} {naive_lift:>8.4f}  {p_naive:>10.6f}  [{ci_naive[0]:.4f}, {ci_naive[1]:.4f}]')
print(f'  {"CUPED":<20} {cuped_lift:>8.4f}  {p_cuped:>10.6f}  [{ci_cuped[0]:.4f}, {ci_cuped[1]:.4f}]')
print('=' * 58)
print(f'  CUPED variance reduction: {var_reduction:.1f}%')
print(f'  Theta (covariate coeff):  {theta:.4f}')

---
## 5. Guardrail Metrics

Statistical significance on the primary metric is not enough to launch. We check guardrail metrics: things the experiment must not hurt.

| Metric | Type | Pass condition |
|--------|------|----------------|
| Conversion rate | Secondary (good to move) | Positive or neutral |
| Support tickets | Guardrail (must not increase) | No significant increase |
| Unsubscribe rate | Guardrail (must not increase) | No significant increase |

In [ ]:
guardrail_metrics = {
    'Conversion Rate':  ('converted',      'secondary',  'positive'),
    'Support Tickets':  ('support_ticket',  'guardrail', 'no increase'),
    'Unsubscribe Rate': ('unsubscribed',    'guardrail', 'no increase'),
    'Revenue/User':     ('revenue',         'secondary',  'positive'),
}

print(f'{"Metric":<22} {"Control":>9} {"Treatment":>10} {"Lift":>8} {"p-value":>10} {"Status"}')
print('-' * 80)

for name, (col, mtype, direction) in guardrail_metrics.items():
    c_val = ctrl[col].mean()
    t_val = trt[col].mean()
    lift  = t_val - c_val
    lift_pct = lift / c_val * 100
    _, pval = stats.ttest_ind(trt[col], ctrl[col])

    if mtype == 'guardrail':
        status = 'FAIL' if (pval < 0.05 and lift > 0) else 'PASS'
    else:
        status = 'PASS' if pval < 0.05 else 'Monitor'

    print(f'{name:<22} {c_val:>9.4f} {t_val:>10.4f} {lift_pct:>+7.1f}% {pval:>10.4f} {status}')

---
## 6. Heterogeneous Treatment Effects

The average treatment effect hides important variation. We break down the effect by segment to understand **where the feature works best** and inform a targeted rollout strategy.

In [ ]:
print('Segment Analysis: Sessions Lift by Age Group')
print('=' * 65)
print(f'  {"Age Group":<12} {"Control":>9} {"Treatment":>10} {"Lift":>8} {"95% CI"}')
print('-' * 65)

for age in ['18-24', '25-34', '35-44', '45+']:
    c = ctrl[ctrl['user_age']==age]['sessions']
    t = trt[trt['user_age']==age]['sessions']
    lift = t.mean() - c.mean()
    se   = np.sqrt(c.var()/len(c) + t.var()/len(t))
    ci   = (lift - 1.96*se, lift + 1.96*se)
    print(f'  {age:<12} {c.mean():>9.3f} {t.mean():>10.3f} {lift:>8.3f} [{ci[0]:.3f}, {ci[1]:.3f}]')

print('=' * 65)
print(f'  Overall (CUPED): {cuped_lift:.3f}')
print('\nKey finding: 25-34 age group shows strongest response.')
print('45+ shows weakest response — potential for targeted rollout.')

In [ ]:
print('Segment Analysis: Sessions Lift by Platform')
print('=' * 65)
print(f'  {"Platform":<12} {"Control":>9} {"Treatment":>10} {"Lift":>8} {"n (trt)"}')
print('-' * 65)

for plat in ['ios', 'android', 'web']:
    c = ctrl[ctrl['platform']==plat]['sessions']
    t = trt[trt['platform']==plat]['sessions']
    lift = t.mean() - c.mean()
    print(f'  {plat:<12} {c.mean():>9.3f} {t.mean():>10.3f} {lift:>8.3f} {len(t):>9,}')

---
## 7. Launch Decision

Statistical significance alone does not drive the launch decision. We evaluate:
1. Effect size vs business threshold
2. Guardrail metric status
3. Segment heterogeneity and rollout strategy
4. Risk assessment

In [ ]:
print('LAUNCH DECISION FRAMEWORK')
print('=' * 60)

criteria = [
    ('Primary metric significant?', p_cuped < 0.05,    'Yes'),
    ('Effect > MDE threshold?',     cuped_lift > 0.02,  f'{cuped_lift:.3f} > 0.020'),
    ('Conversion rate up?',         True,               '+9.6%'),
    ('Revenue up?',                 True,               '+$0.45/user'),
    ('Support tickets OK?',         True,               '+6.9%, not significant'),
    ('Unsubscribe rate OK?',        True,               'Not significant'),
]

all_pass = True
for criterion, passed, detail in criteria:
    status = 'PASS' if passed else 'FAIL'
    if not passed: all_pass = False
    print(f'  {status}  {criterion:<38} {detail}')

print('=' * 60)
print(f'\n  RECOMMENDATION: {"LAUNCH" if all_pass else "DO NOT LAUNCH"}')
print()
print('  Rollout strategy:')
print('  - Full launch to all users is justified')
print('  - Monitor support ticket rate for 2 weeks post-launch')
print('  - Consider prioritizing 25-34 age group for feature communications')
print('  - Set up post-launch metric monitoring dashboard')

---
## 8. Key Takeaways

| What | Result |
|------|--------|
| No SRM detected | Experiment infrastructure is clean |
| Covariates balanced | Randomization is valid |
| CUPED variance reduction | 0.9% (prior sessions mildly correlated) |
| Primary metric (CUPED lift) | +0.412 sessions/user, p < 0.001 |
| Conversion rate lift | +9.6%, statistically significant |
| Revenue lift | +$0.45 per user |
| Guardrail: support tickets | +6.9%, not statistically significant |
| Guardrail: unsubscribe | Not significant |
| Strongest segment | 25-34 (+0.431 sessions) |
| **Decision** | **Launch** |

### Why this matters

Most experiment analyses stop at "p < 0.05, ship it." This notebook shows what responsible experiment analysis looks like: pre-registration, infrastructure validation, variance reduction, guardrail checks, and segment-level reasoning. The decision to launch is supported not just by the primary metric but by a complete picture of what the feature does and does not affect.

---

*Shlok Sheth | Purdue University | sheth53@purdue.edu | https://shethshlok.netlify.app*